In [ ]:
%load_ext autoreload
%autoreload 2
# %matplotlib qt
from ma.mtgp import MTGP, HR_SIGNALS, RR_SIGNALS
import matplotlib.pyplot as plt
import pandas as pd
pd.set_option('display.max_columns', None)
parti_no = 1
mtgp = MTGP(parti_no=parti_no)

# Get Peaks:

In [ ]:
segment_no = 3
def get_peak_color(peak_label:int) -> str:
    match peak_label:
        case 0:
            return "green"
        case 1:
            return "orange"
        case 2:
            return "red"
        case -1:
            return "purple"


df_parti = mtgp.df_parti_segmented_hr[segment_no]
peaks_hr = mtgp.peaks_parti["HR"]

df_ecg = df_parti["ecg3"]
df_ppg = df_parti["ppg1"]
peaks_ecg = peaks_hr["ecg3"][0][segment_no]
peak_ecg_label = peaks_hr["ecg3"][1][segment_no]

peaks_ppg = peaks_hr["ppg1"][0][segment_no]
peak_ppg_label = peaks_hr["ppg1"][1][segment_no]

fig, ax = plt.subplots(ncols = 2, figsize = (29,9))
ax[0].plot(df_ecg, color = "blue", label = "cECG")
ax[0].scatter(df_ecg.index[peaks_ecg], df_ecg.values[peaks_ecg], color = get_peak_color(peak_ecg_label))
ax2 = ax[0].twinx()
ax2.plot(df_ppg, color = "green", label = "rPPG")
ax2.scatter(df_ppg.index[peaks_ppg], df_ppg.values[peaks_ppg], color = get_peak_color(peak_ppg_label))

ax[0].grid()
ax[0].legend()
########################
df_parti = mtgp.df_parti_segmented_hr[segment_no+1]
df_ecg = df_parti["ecg3"]
df_ppg = df_parti["ppg1"]
peaks_ecg = peaks_hr["ecg3"][0][segment_no+1]
peak_ecg_label = peaks_hr["ecg3"][1][segment_no+1]
peaks_ppg = peaks_hr["ppg1"][0][segment_no+1]
peak_ppg_label = peaks_hr["ppg1"][1][segment_no+1]


ax[1].plot(df_ecg, color = "blue", label = "cECG")
ax[1].scatter(df_ecg.index[peaks_ecg], df_ecg.values[peaks_ecg], color = get_peak_color(peak_ecg_label))
ax2 = ax[1].twinx()
ax2.plot(df_ppg, color = "green", label = "rPPG")
ax2.scatter(df_ppg.index[peaks_ppg], df_ppg.values[peaks_ppg], color = get_peak_color(peak_ppg_label))

ax[1].grid()
ax[1].legend()

# Get HR / RR from the Best Signal:

### Merged HR/RR DataFrames:
- Merged and interpolated based on the reference signal !

In [ ]:
merged_hr = mtgp.hr_merged

hr_ecg1 = mtgp.hr_rr_parti["HR"]["ecg1"]
hr_ref = mtgp.hr_rr_ref["HR"]["hr"]
hr_ecg1_merged = merged_hr[["hr_ecg1", "sq_labels_ecg1"]]

fig, ax = plt.subplots(figsize = (29,9))
# Define colors for each label
colors = {0: "green", 1: "orange", 2: "red", -1: "purple"}

# Plot each subset of data by label
for label, color in colors.items():
    subset_ecg1 = hr_ecg1[hr_ecg1["sq_labels"] == label]
    subset_merged = hr_ecg1_merged[hr_ecg1_merged["sq_labels_ecg1"] == label]
    ax.scatter(subset_ecg1.index, subset_ecg1["hr"], color=color)
    ax.scatter(subset_merged.index, subset_merged["hr_ecg1"], color=color)

ax.plot(hr_ref.index, hr_ref, color = "orange", marker = "o", label = "HR - REF", linewidth= 3.0)
ax.plot(hr_ecg1.index, hr_ecg1["hr"], color = "blue", label = "HR - cECG1")
ax.plot(hr_ecg1_merged.index, hr_ecg1_merged["hr_ecg1"], color = "green", label = "HR - merged")


## Legend:
ax.scatter([],[], color = "green", marker = "o", label = "Good Quality")
ax.scatter([],[], color = "orange", marker = "o", label = "Bad Quality")
ax.scatter([],[], color = "red", marker = "o", label = "Noisy Quality")
ax.scatter([],[], color = "purple", marker = "o", label = "Unknown Quality")

ax.grid()
# ax2.grid()
fig.legend()

### Select Best HR/RR data points to create the Final HR/RR Signal

In [ ]:
%matplotlib qt

In [ ]:
hr_ref = mtgp.hr_rr_ref["HR"]["hr"]
hr_fused = mtgp.hr_fused

fig, ax = plt.subplots(figsize = (29,9), nrows =2, sharex=True)
ax[0].plot(hr_ref.index, hr_ref, color = "orange", marker = "o", label = "HR - REF", linewidth= 3.0)
ax[0].plot(hr_fused.index, hr_fused["hr_fused"], color = "green", label = "HR - FUSED", linewidth= 3.0)

ma_indices = hr_fused["ma_labels_fused"].index[hr_fused["ma_labels_fused"] == 1]
for ma_idx in ma_indices:
    iloc_position = min(hr_fused.index.get_loc(ma_idx), len(hr_fused) -2 )
    next_idx = hr_fused.index[iloc_position + 1]
    ax[0].axvspan(
        ma_idx,
        next_idx,
        color="lightgray",
        alpha=0.8,
    )

# Define colors for each label
colors = {0: "green", 1: "orange", 2: "red", -1: "purple"}

for signal in HR_SIGNALS:
    hr_parti = mtgp.hr_merged[[f"hr_{signal}", f"sq_labels_{signal}"]]
    # Plot each subset of data by label
    for label, color in colors.items():
        subset_fused = hr_fused[hr_fused["sq_labels_fused"] == label]
        subset = hr_parti[hr_parti[f"sq_labels_{signal}"] == label]
        ax[0].scatter(subset_fused.index, subset_fused["hr_fused"], color=color)
    #     ax[0].scatter(subset.index, subset[f"hr_{signal}"], color=color, alpha = 0.2)
    # ax[0].plot(hr_parti.index, hr_parti[f"hr_{signal}"], label = f"HR - {signal}", alpha = 0.2)

## Legend:
ax[0].scatter([],[], color = "green", marker = "o", label = "Good Quality")
ax[0].scatter([],[], color = "orange", marker = "o", label = "Bad Quality")
ax[0].scatter([],[], color = "red", marker = "o", label = "Noisy Quality")
ax[0].scatter([],[], color = "purple", marker = "o", label = "Unknown Quality")

ax[0].grid()
# ax2.grid()

############################### RR ####################################
rr_ref = mtgp.hr_rr_ref["RR"]["rr"]
rr_fused = mtgp.rr_fused

ax[1].plot(rr_ref.index, rr_ref, color = "orange", marker = "o", label = "RR - REF", linewidth= 3.0)
ax[1].plot(rr_fused.index, rr_fused["rr_fused"], color = "green", label = "RR - FUSED", linewidth= 3.0)

ma_indices = rr_fused["ma_labels_fused"].index[rr_fused["ma_labels_fused"] == 1]
for ma_idx in ma_indices:
    iloc_position = min(rr_fused.index.get_loc(ma_idx), len(rr_fused) -2 )
    next_idx = rr_fused.index[iloc_position + 1]
    ax[1].axvspan(
        ma_idx,
        next_idx,
        color="lightgray",
        alpha=0.8,
    )

# Define colors for each label
colors = {0: "green", 1: "orange", 2: "red", -1: "purple"}

for signal in RR_SIGNALS:
    rr_parti = mtgp.rr_merged[[f"rr_{signal}", f"sq_labels_{signal}"]]
    # Plot each subset of data by label
    for label, color in colors.items():
        subset_fused = rr_fused[rr_fused["sq_labels_fused"] == label]
        subset = rr_parti[rr_parti[f"sq_labels_{signal}"] == label]
        ax[1].scatter(subset_fused.index, subset_fused["rr_fused"], color=color)
    #     ax[1].scatter(subset.index, subset[f"rr_{signal}"], color=color, alpha = 0.2)
    # ax[1].plot(rr_parti.index, rr_parti[f"rr_{signal}"], label = f"RR - {signal}", alpha = 0.2)

## Legend:
ax[1].scatter([],[], color = "green", marker = "o", label = "Good Quality")
ax[1].scatter([],[], color = "orange", marker = "o", label = "Bad Quality")
ax[1].scatter([],[], color = "red", marker = "o", label = "Noisy Quality")
ax[1].scatter([],[], color = "purple", marker = "o", label = "Unknown Quality")

ax[1].grid()
fig.legend()

In [ ]:
rr_fused

### Merge HR Fused & RR Fused 

> HOW TO PROCEED ?

In [ ]:
import numpy as np
import GPy
import matplotlib.pyplot as plt

df_rr_train = mtgp.rr_fused.loc[1109:]
df_hr_train = mtgp.hr_fused.loc[1103:1275]
df_hr_rest = mtgp.hr_fused.loc[1277:]

df_hr_ma = mtgp.hr_fused.loc[1083.5:1089]
df_rr_ma = mtgp.rr_fused.loc[1095:1110]
df_hr_ma2 = mtgp.hr_fused.loc[1274.5:1277.5]

fig, ax = plt.subplots()
ax.plot(df_hr_train.index, df_hr_train["hr_fused"], color = "green", linewidth = 3.0, marker = "o", label = "HR - Train")
ax.plot(df_hr_rest.index, df_hr_rest["hr_fused"], color = "green", linewidth = 3.0, marker = "o")
ax.plot(df_hr_ma2.index, df_hr_ma2["hr_fused"], color = "red", linewidth = 1.0, marker = "o", label= "HR - MA")

ax.plot(df_rr_train.index, df_rr_train["rr_fused"], color = "blue", linewidth = 3.0, marker = "o", label = "RR - Train")
ax.plot(df_rr_ma.index, df_rr_ma["rr_fused"], color = "red", linewidth = 1.0, marker = "o", label= "RR - MA")

ax.grid()
ax.legend()

In [ ]:
time_hr = df_hr_train.index.to_numpy()[:, None]
time_rest = df_hr_rest.index.to_numpy()[:, None]
time_hr = np.concatenate([time_hr, time_rest])
time_rr = df_rr_train.index.to_numpy()[:, None]

hr_data = df_hr_train["hr_fused"].values[:, None]
hr_rest = df_hr_rest["hr_fused"].values[:, None]
hr_data = np.concatenate([hr_data, hr_rest])
rr_data = df_rr_train["rr_fused"].values[:, None]

# Stack time points and label them per X_hr (0 for HR, 1 for RR)
X_hr = np.hstack([time_hr, np.zeros_like(time_hr)])  # Task label 0 for HR
X_rr = np.hstack([time_rr, np.ones_like(time_rr)])   # Task label 1 for RR

# Reshape outputs to 2D arrays
Y_hr = hr_data.reshape(-1, 1)
Y_rr = rr_data.reshape(-1, 1)

# Define the multi-output kernel with coregionalization
kernel = GPy.kern.RBF(1) ** GPy.kern.Coregionalize(1, output_dim=2)

# Initialize the MTGP model
mtgp_model = GPy.models.GPCoregionalizedRegression([X_hr, X_rr], [Y_hr, Y_rr], kernel=kernel)

# Optimize the model
mtgp_model.optimize()


In [ ]:
time1 = df_hr_train.index.to_numpy()[:, None]
time2 = df_hr_ma2.index.to_numpy()[:,None]
time3 = df_hr_rest.index.to_numpy()[:, None]
time_test_hr = np.concatenate([time1, time2, time3])

time1 = df_rr_ma.index.to_numpy()[:, None]
time2 = df_rr_train.index.to_numpy()[:,None]
time_test_rr = np.concatenate([time1, time2])

# Prepare input matrices for each task
X_test_hr = np.hstack([time_test_hr, np.zeros_like(time_test_hr)])  # Task label 0
X_test_rr = np.hstack([time_test_rr, np.ones_like(time_test_rr)])   # Task label 1

# Prediction metadata for each task
Y_metadata_hr = {'output_index': X_test_hr[:, 1].astype(int).reshape(-1, 1)}
Y_metadata_rr = {'output_index': X_test_rr[:, 1].astype(int).reshape(-1, 1)}

# Predict for each task
Y_pred_hr, Y_var_hr = mtgp_model.predict(X_test_hr, Y_metadata=Y_metadata_hr)
Y_pred_rr, Y_var_rr = mtgp_model.predict(X_test_rr, Y_metadata=Y_metadata_rr)


# Plot results

# Heart Rate predictions
ax.plot(time_test_hr, Y_pred_hr, 'b--', label='Predicted HR')
ax.fill_between(
    time_test_hr.flatten(),
    Y_pred_hr.flatten() - 1.96 * np.sqrt(Y_var_hr.flatten()),
    Y_pred_hr.flatten() + 1.96 * np.sqrt(Y_var_hr.flatten()),
    color="blue",
    alpha=0.2,
)

# Respiratory Rate predictions
plt.plot(time_test_rr, Y_pred_rr, 'g--', label='Predicted RR')
plt.fill_between(
    time_test_rr.flatten(),
    Y_pred_rr.flatten() - 1.96 * np.sqrt(Y_var_rr.flatten()),
    Y_pred_rr.flatten() + 1.96 * np.sqrt(Y_var_rr.flatten()),
    color="green",
    alpha=0.2,
)

plt.xlabel("Time")
plt.ylabel("Rate")
plt.legend()
plt.show()
